# 📈 Hands-On: Zeitreihenprognose mit Chronos
### Zero-Shot Forecasting zum Selbermachen

Gleiches Format wie das Prophet-Hands-on: **Setup, Daten und Plots sind vorgegeben** (einfach ausführen). Du schreibst nur den Modell-Teil (Aufgabe 1 & 2).

Datensatz: Verkäufe der **„SkyDrive X1 Pro" SSD** (synthetisch: Trend + Wochenmuster + Black-Friday-Peak).

---
## 0 · Setup *(vorgegeben – einfach ausführen)*

In [ ]:
# !pip install chronos-forecasting scikit-learn --quiet
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from chronos import BaseChronosPipeline
np.random.seed(0); torch.manual_seed(0)

def print_metrics(y_true, y_pred, label='Modell'):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f'{label:<28} RMSE={rmse:8.1f}  MAE={mae:8.1f}  MAPE={mape:5.1f}%')
    return {'label': label, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

def plot_forecast(y_train, y_true, median, lo, hi, label, n_context=90):
    H = len(y_true)
    xh = np.arange(-min(n_context, len(y_train)), 0); xf = np.arange(0, H)
    plt.figure(figsize=(13, 4))
    plt.plot(xh, y_train[-len(xh):], color='gray', label='Historie')
    plt.plot(xf, y_true, color='black', marker='.', label='Test (echt)')
    plt.plot(xf, median, color='C1', label=f'{label} - Median')
    plt.fill_between(xf, lo, hi, color='C1', alpha=0.25, label='10-90%-Band')
    plt.axvline(0, color='k', ls=':'); plt.legend(); plt.title(label)
    plt.tight_layout(); plt.show()

## 0.1 · Datengenerierung *(vorgegeben – einfach ausführen)*

Erzeugt die SkyDrive-Zeitreihe (Trend + Wochenmuster + Black-Friday-Spitze) und teilt in Train/Test. Der Test-Zeitraum enthält bewusst einen **Black Friday** – darauf zielt Aufgabe 2.

In [ ]:
def make_skydrive(seed=7):
    rng = np.random.default_rng(seed)
    dates = pd.date_range('2022-01-01', '2023-12-31', freq='D')
    t = np.arange(len(dates))
    dow = np.asarray(dates.dayofweek)                      # als NumPy-Array statt pandas-Index
    base = 200 + 0.15 * t + 30 * np.sin(2 * np.pi * dow / 7)
    sales = np.asarray(base, dtype=float) + rng.normal(0, 12, len(dates))
    for yr in [2022, 2023]:
        fr = dates[(dates.year == yr) & (dates.month == 11) & (dates.dayofweek == 4)]
        bf = fr[3]  # 4. Freitag im November = Black Friday
        for d, boost in [(bf - pd.Timedelta(days=1), 0.5), (bf, 2.5),
                         (bf + pd.Timedelta(days=1), 1.0), (bf + pd.Timedelta(days=3), 1.2)]:
            i = dates.get_indexer([d])[0]
            if i >= 0:
                sales[i] += base[i] * boost
    return pd.DataFrame({'ds': dates, 'y': np.clip(sales, 0, None)})

def black_friday_flag(dates):
    dates = pd.DatetimeIndex(dates)              # akzeptiert Series ODER Index
    flag = np.zeros(len(dates), float)
    for yr in np.unique(dates.year):
        fr = dates[(dates.year == yr) & (dates.month == 11) & (dates.dayofweek == 4)]
        if len(fr) >= 4:
            win = pd.date_range(fr[3] - pd.Timedelta(days=1), fr[3] + pd.Timedelta(days=3))
            flag[dates.isin(win)] = 1.0
    return flag

df = make_skydrive()
df['black_friday'] = black_friday_flag(df['ds'])

H = 49  # letzte 49 Tage als Test (enthält Black Friday 2023)
y = df['y'].to_numpy(float)
y_train, y_test = y[:-H], y[-H:]
print(f'{len(df)} Tage gesamt | Train {len(y_train)} | Test {len(y_test)}')
print('Test-Zeitraum:', df['ds'].iloc[-H].date(), 'bis', df['ds'].iloc[-1].date())

## 0.2 · Daten visualisieren *(vorgegeben – einfach ausführen)*

In [ ]:
plt.figure(figsize=(13, 4))
plt.plot(df['ds'], df['y'], color='C0', lw=1)
bf = df[df['black_friday'] == 1]
plt.scatter(bf['ds'], bf['y'], color='red', zorder=5, label='Black Friday')
plt.title('SkyDrive X1 Pro - Tagesumsatz'); plt.legend(); plt.tight_layout(); plt.show()

---
## 🎯 Aufgabe 1 – Naiver Zero-Shot-Forecast

Lade ein Chronos-Bolt-Modell und sage die Testperiode vorher – **ohne jede Zusatzinfo**. Gegenstück zum „naiven Prophet-Modell ohne Feiertage".

**Schritte:**
1. Modell laden: `BaseChronosPipeline.from_pretrained('amazon/chronos-bolt-small', device_map='cpu')`
2. Vorhersagen: `predict_quantiles(inputs=torch.tensor(y_train, dtype=torch.float32), prediction_length=H, quantile_levels=[0.1, 0.5, 0.9])` (das erste Argument heißt `inputs`)
3. Ergebnisse speichern als **`median1`**, **`lo1`**, **`hi1`** (über `quantiles[0, :, 0/1/2].numpy()`)
4. Bewerten: `print_metrics(y_test, median1, 'Aufgabe 1: naiv')`

In [ ]:
# TODO Aufgabe 1 – dein Code hier:
# pipe = ...
# quantiles, mean = pipe.predict_quantiles(...)
# median1, lo1, hi1 = ...
# print_metrics(y_test, median1, 'Aufgabe 1: naiv')


## 📊 Ergebnis Aufgabe 1 visualisieren *(vorgegeben – einfach ausführen)*

In [ ]:
plot_forecast(y_train, y_test, median1, lo1, hi1, 'Aufgabe 1: naiv (zero-shot)')

---
## 🎯 Aufgabe 2 – Modell verbessern

Wähle **eine** Variante und speichere die Punktprognose als **`median2`** (sowie `lo2`, `hi2` falls vorhanden).

**2a (Pflicht, einfach):** größeres Modell `amazon/chronos-bolt-base` ODER mehr Kontext.

**2b (Bonus, Chronos-2):** Black Friday als **Kovariate**. Spalte `df['black_friday']` ist schon vorhanden. Gegenstück zum „Experten-Modell" bei Prophet.

```python
# Skizze 2b:
from chronos import Chronos2Pipeline
pipe2 = Chronos2Pipeline.from_pretrained('amazon/chronos-2', device_map='cpu')
context_df = pd.DataFrame({'id': 'skydrive', 'timestamp': df['ds'][:-H].values,
                           'target': y_train, 'black_friday': df['black_friday'].to_numpy()[:-H]})
future_df  = pd.DataFrame({'id': 'skydrive', 'timestamp': df['ds'][-H:].values,
                           'black_friday': df['black_friday'].to_numpy()[-H:]})
pred_df = pipe2.predict_df(context_df, future_df=future_df, prediction_length=H,
            quantile_levels=[0.1, 0.5, 0.9], id_column='id', timestamp_column='timestamp', target='target')
median2 = pred_df['0.5'].to_numpy()
```

In [ ]:
# TODO Aufgabe 2 (a oder b) – dein Code hier:
# median2, lo2, hi2 = ...
# print_metrics(y_test, median2, 'Aufgabe 2: verbessert')


## 📊 Showdown: naiv vs. verbessert *(vorgegeben – einfach ausführen)*

In [ ]:
print_metrics(y_test, median1, 'Aufgabe 1: naiv')
print_metrics(y_test, median2, 'Aufgabe 2: verbessert')

xf = np.arange(H)
plt.figure(figsize=(13, 4))
plt.plot(xf, y_test, color='black', marker='.', label='Test (echt)')
plt.plot(xf, median1, color='C0', label='naiv')
plt.plot(xf, median2, color='C1', label='verbessert')
plt.title('Showdown: naiv vs. verbessert'); plt.legend(); plt.tight_layout(); plt.show()

---
## 💬 Diskussionsfragen

1. Chronos braucht **kein** `fit()` und keine Feiertage – warum funktioniert es trotzdem?
2. Was hat bei *dir* mehr gebracht: mehr Kontext, ein größeres Modell oder die Kovariate?
3. Wann würdest du Prophet bevorzugen, wann Chronos? (Erklärbarkeit, Rechenzeit, Datenmenge, Domänenwissen)